In [1]:
import pandas as pd 

#hayek_citations_df = pd.read_csv("../data/processed/hayek_citations.csv")
#mises_citations_df = pd.read_csv("../data/processed/mises_citations.csv")

hayek_citations_df = pd.read_csv("../data/processed/hayek_paragraph_citations.csv")
mises_citations_df = pd.read_csv("../data/processed/mises_paragraph_citations.csv")

In [2]:
mises_citations_df.head()

,paragraph_id,paragraph,paragraph_keywords,sentence_keywords,sentence_id,human_action_chapter_name,human_action_chapter_number,human_action_part_name,human_action_part_number
0,_22Yv29R,As economic man journeyed into neoclassical ec...,"['neoclassical', 'economics', 'wealth', 'human...",['human action'],['_MxKnfHC'],['WHOLE'],['WHOLE'],['WHOLE'],['WHOLE']
1,_22zyRMQ,"Thus, it seems as if a thoroughgoing integrati...","['uncertainty', 'price', 'the market', 'econom...",['NO-MATCH'],"['_qBKNcCm', '_XQHxBBz']",['Chapter 14: The Scope and Method of Catallat...,"['14', '16']",['Part 4: Catallatics'],['4']
2,_23fQSu6,"63 Simmel (1978), p. 99. 64Op. cit., p. 97. 65...","['catallaxy', 'social order', 'cooperation', '...",['sociology'],['_Kj6czt7'],['Chapter 2: The Epistemological Problems of t...,['2'],['Part 1: Human Action'],['1']
3,_24rBZqf,Because phenomenological descriptions cannot b...,"['life-world', 'theory and history', 'abstract...",['theory and history'],"['_9783h2g', '_mb5StsQ']",['WHOLE'],['WHOLE'],['WHOLE'],['WHOLE']
4,_25wnbvA,"En appendice, comme pour ne pas leur ""voler la...","['kirzner', 'lachmann', 'perspectives', 'infor...","['kirzner', 'lachmann']",['_YR55R3b'],['WHOLE'],['WHOLE'],['WHOLE'],['WHOLE']


In [3]:
hayek_citations_df[["paragraph", "sentence_keywords", "paragraph_keywords"]].head()

,paragraph,sentence_keywords,paragraph_keywords
0,The problem caused by the distortion of the in...,"['expectations', 'interest']","['interest', 'time', 'savings', 'consumption',..."
1,of what is seemingly non-rational (habitual) b...,['NO-MATCH'],"['calculation', 'social rules', 'rational reco..."
2,The ability of organizations to adapt in chang...,['networks'],"['the austrian school', 'kirzner', 'lachmann',..."
3,But if capital goods can extend the mental rea...,"['behavioral', 'norms', 'institutions', 'coord...","['capital', 'goods', 'information', 'social le..."
4,There is still scope for the utilitarian yards...,"['rule of law', 'regulation', 'markets', 'know...","['scope', 'rule of law', 'regulation', 'market..."


In [4]:
import ast
import pandas as pd

def ensure_list(x):
    # já é lista → retorna
    if isinstance(x, list):
        return x
    
    # nulo → lista vazia
    if pd.isna(x):
        return []
    
    # string → tenta converter
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, list) else [val]
        except:
            # fallback: trata como único item
            return [x]
    
    # qualquer outro tipo → encapsula
    return [x]


# aplicar nos dois dataframes
mises_citations_df["paragraph_keywords"] = mises_citations_df["paragraph_keywords"].apply(ensure_list)
hayek_citations_df["paragraph_keywords"] = hayek_citations_df["paragraph_keywords"].apply(ensure_list)

Contagem Geral 

In [5]:
import pandas as pd
import ast

# ==================================================
# 0️⃣ Garantir que keywords são listas
# ==================================================
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, list) else [val]
        except:
            return [x]
    return [x]


for df in [mises_citations_df, hayek_citations_df]:
    df["paragraph_keywords"] = df["paragraph_keywords"].apply(ensure_list)
    df["sentence_keywords"] = df["sentence_keywords"].apply(ensure_list)


# ==================================================
# 🔧 Função genérica (sentence / paragraph)
# ==================================================
def explode_keywords(df, id_col, keyword_col, source_name):
    temp = df[[id_col, keyword_col]].copy()
    
    temp = temp.explode(keyword_col)
    temp = temp.dropna(subset=[keyword_col])
    temp = temp.drop_duplicates([id_col, keyword_col])
    
    temp[source_name] = 1
    return temp


def build_keyword_df(df1, df2, id_col, keyword_col):
    kw1 = explode_keywords(df1, id_col, keyword_col, "mises")
    kw2 = explode_keywords(df2, id_col, keyword_col, "hayek")
    
    merged = pd.merge(
        kw1, kw2,
        on=[id_col, keyword_col],
        how="outer"
    )
    
    merged["mises"] = merged["mises"].fillna(0)
    merged["hayek"] = merged["hayek"].fillna(0)
    
    # categorias
    merged["mises_only"] = ((merged["mises"] == 1) & (merged["hayek"] == 0)).astype(int)
    merged["hayek_only"] = ((merged["mises"] == 0) & (merged["hayek"] == 1)).astype(int)
    merged["both"] = ((merged["mises"] == 1) & (merged["hayek"] == 1)).astype(int)
    
    # agregação
    keyword_df = (
        merged
        .groupby(keyword_col)
        .agg(
            mises_only_count=("mises_only", "sum"),
            hayek_only_count=("hayek_only", "sum"),
            mises_and_hayek_count=("both", "sum"),
        )
        .reset_index()
    )
    
    # total
    keyword_df["total_count"] = (
        keyword_df["mises_only_count"] +
        keyword_df["hayek_only_count"] +
        keyword_df["mises_and_hayek_count"]
    )
    
    # evitar divisão por zero
    keyword_df["total_count"] = keyword_df["total_count"].replace(0, 1)
    
    # probabilidades condicionais
    keyword_df["p_mises_only"] = (
        keyword_df["mises_only_count"] / keyword_df["total_count"]
    )
    
    keyword_df["p_hayek_only"] = (
        keyword_df["hayek_only_count"] / keyword_df["total_count"]
    )
    
    keyword_df["p_both"] = (
        keyword_df["mises_and_hayek_count"] / keyword_df["total_count"]
    )
    
    return keyword_df.sort_values("total_count", ascending=False)


# ==================================================
# 1️⃣ Paragraph-level
# ==================================================
keywords_paragraph_df = build_keyword_df(
    mises_citations_df,
    hayek_citations_df,
    "paragraph_id",
    "paragraph_keywords"
).rename(columns={"paragraph_keywords": "keyword"})


# ==================================================
# 2️⃣ Sentence-level
# ==================================================
keywords_sentence_df = build_keyword_df(
    mises_citations_df,
    hayek_citations_df,
    "sentence_id",
    "sentence_keywords"
).rename(columns={"sentence_keywords": "keyword"})


# ==================================================
# ✅ OUTPUTS
# ==================================================
print("Paragraph-level:")
print(keywords_paragraph_df.head())

print("\nSentence-level:")
print(keywords_sentence_df.head())

Paragraph-level:
         keyword  mises_only_count  hayek_only_count  mises_and_hayek_count  \
939      process               812               846                    249   
658    knowledge               607              1058                    239   
65      austrian               836               611                    369   
1182        time               825               753                    181   
1171  the market               909               582                    228   

      total_count  p_mises_only  p_hayek_only    p_both  
939          1907      0.425800      0.443629  0.130572  
658          1904      0.318803      0.555672  0.125525  
65           1816      0.460352      0.336454  0.203194  
1182         1759      0.469016      0.428084  0.102899  
1171         1719      0.528796      0.338569  0.132635  

Sentence-level:
       keyword  mises_only_count  hayek_only_count  mises_and_hayek_count  \
1     NO-MATCH               914               705                

In [6]:
# ==================================================
# 🔧 Função para gerar contagens por unidade
# ==================================================
def build_counts(df1, df2, id_col, keyword_col):
    kw1 = explode_keywords(df1, id_col, keyword_col, "mises")
    kw2 = explode_keywords(df2, id_col, keyword_col, "hayek")
    
    merged = pd.merge(
        kw1, kw2,
        on=[id_col, keyword_col],
        how="outer"
    )
    
    merged["mises"] = merged["mises"].fillna(0)
    merged["hayek"] = merged["hayek"].fillna(0)
    
    # categorias
    merged["mises_only"] = ((merged["mises"] == 1) & (merged["hayek"] == 0))
    merged["hayek_only"] = ((merged["mises"] == 0) & (merged["hayek"] == 1))
    merged["both"] = ((merged["mises"] == 1) & (merged["hayek"] == 1))
    
    # contar IDs únicos por categoria
    counts = {
        "mises_only": merged.loc[merged["mises_only"], id_col].nunique(),
        "hayek_only": merged.loc[merged["hayek_only"], id_col].nunique(),
        "mises_and_hayek": merged.loc[merged["both"], id_col].nunique(),
    }
    
    return counts


# ==================================================
# 1️⃣ Sentence counts
# ==================================================
sentence_counts = build_counts(
    mises_citations_df,
    hayek_citations_df,
    "sentence_id",
    "sentence_keywords"
)


# ==================================================
# 2️⃣ Paragraph counts
# ==================================================
paragraph_counts = build_counts(
    mises_citations_df,
    hayek_citations_df,
    "paragraph_id",
    "paragraph_keywords"
)


# ==================================================
# 3️⃣ Construir counts_df
# ==================================================
counts_df = pd.DataFrame({
    "sentence_count": sentence_counts,
    "paragraph_count": paragraph_counts
})


# garantir ordem das linhas
counts_df = counts_df.loc[
    ["hayek_only", "mises_only", "mises_and_hayek"]
]


# ==================================================
# ✅ Resultado final
# ==================================================
counts_df

,sentence_count,paragraph_count
hayek_only,4419,3915
mises_only,5575,5071
mises_and_hayek,435,939


Estatísticas Keywords

In [7]:
keywords_paragraph_df = keywords_paragraph_df[keywords_paragraph_df["total_count"] > 50]
keywords_sentence_df = keywords_sentence_df[keywords_sentence_df["total_count"] > 50]

In [8]:
keywords_paragraph_df.sort_values(by='total_count', ascending=False)

,keyword,mises_only_count,hayek_only_count,mises_and_hayek_count,total_count,p_mises_only,p_hayek_only,p_both
939,process,812,846,249,1907,0.425800,0.443629,0.130572
658,knowledge,607,1058,239,1904,0.318803,0.555672,0.125525
65,austrian,836,611,369,1816,0.460352,0.336454,0.203194
1182,time,825,753,181,1759,0.469016,0.428084,0.102899
1171,the market,909,582,228,1719,0.528796,0.338569,0.132635
...,...,...,...,...,...,...,...,...
814,neoliberalism,13,32,8,53,0.245283,0.603774,0.150943
245,creative destruction,31,19,3,53,0.584906,0.358491,0.056604
710,mainstream economics,21,25,6,52,0.403846,0.480769,0.115385
206,consistency,15,32,4,51,0.294118,0.627451,0.078431


Calculando Lifts

In [9]:
# ==================================================
# 1️⃣ Totais globais (parágrafos)
# ==================================================
total_paragraphs = counts_df["paragraph_count"].sum()

total_mises = counts_df.loc["mises_only", "paragraph_count"]
total_hayek = counts_df.loc["hayek_only", "paragraph_count"]
total_both = counts_df.loc["mises_and_hayek", "paragraph_count"]

# ==================================================
# 2️⃣ Probabilidade global da keyword
# ==================================================
keywords_paragraph_df["p_keyword"] = (
    keywords_paragraph_df["total_count"] / total_paragraphs
)

# ==================================================
# 3️⃣ Probabilidades condicionais
# ==================================================
keywords_paragraph_df["p_kw_mises"] = (
    keywords_paragraph_df["mises_only_count"] / total_mises
)

keywords_paragraph_df["p_kw_hayek"] = (
    keywords_paragraph_df["hayek_only_count"] / total_hayek
)

keywords_paragraph_df["p_kw_both"] = (
    keywords_paragraph_df["mises_and_hayek_count"] / total_both
)

# ==================================================
# 4️⃣ Lift
# ==================================================
keywords_paragraph_df["lift_mises_only"] = (
    keywords_paragraph_df["p_kw_mises"] / keywords_paragraph_df["p_keyword"]
)

keywords_paragraph_df["lift_hayek_only"] = (
    keywords_paragraph_df["p_kw_hayek"] / keywords_paragraph_df["p_keyword"]
)

keywords_paragraph_df["lift_mises_and_hayek"] = (
    keywords_paragraph_df["p_kw_both"] / keywords_paragraph_df["p_keyword"]
)

# ==================================================
# 5️⃣ (Opcional) limpar colunas intermediárias
# ==================================================
keywords_paragraph_df = keywords_paragraph_df.drop(
    columns=["p_keyword", "p_kw_mises", "p_kw_hayek", "p_kw_both"]
)

# ==================================================
# 6️⃣ Ordenar por lift (exemplo)
# ==================================================
keywords_paragraph_df = keywords_paragraph_df.sort_values(
    by="lift_mises_and_hayek", ascending=False
)

keywords_paragraph_df.head()

,keyword,mises_only_count,hayek_only_count,mises_and_hayek_count,total_count,p_mises_only,p_hayek_only,p_both,lift_mises_only,lift_hayek_only,lift_mises_and_hayek
110,bust,43,25,47,115,0.373913,0.217391,0.408696,0.731825,0.551113,4.319813
1103,socialist calculation,44,34,48,126,0.349206,0.269841,0.380952,0.683469,0.684080,4.026573
4,abct,46,32,44,122,0.377049,0.262295,0.360656,0.737964,0.664950,3.812043
773,monetary expansion,16,23,20,59,0.271186,0.389831,0.338983,0.530768,0.988268,3.582968
470,garrison,39,96,69,204,0.191176,0.470588,0.338235,0.374172,1.192998,3.575064


Exportando Excel

In [10]:
import pandas as pd
import os
import re

def export_to_excel(keyword, 
                    mises_citations_df, 
                    hayek_citations_df):

    # Sanitizar keyword
    safe_keyword = re.sub(r'[^a-zA-Z0-9_-]', '_', keyword)
    keyword_lower = keyword.lower()

    # Filtrar
    mises_filtered = mises_citations_df[
        mises_citations_df['paragraph'].fillna('').str.lower().str.contains(keyword_lower)
    ]

    hayek_filtered = hayek_citations_df[
        hayek_citations_df['paragraph'].fillna('').str.lower().str.contains(keyword_lower)
    ]

    # Garantir coluna paragraph_id existe
    if 'paragraph_id' not in mises_filtered.columns:
        mises_filtered['paragraph_id'] = None
    if 'paragraph_id' not in hayek_filtered.columns:
        hayek_filtered['paragraph_id'] = None

    # Interseção
    common_ids = set(mises_filtered['paragraph_id']).intersection(
        set(hayek_filtered['paragraph_id'])
    )

    mises_and_hayek = pd.concat([
        mises_filtered[mises_filtered['paragraph_id'].isin(common_ids)],
        hayek_filtered[hayek_filtered['paragraph_id'].isin(common_ids)]
    ], ignore_index=True)

    mises_only = mises_filtered[~mises_filtered['paragraph_id'].isin(common_ids)]
    hayek_only = hayek_filtered[~hayek_filtered['paragraph_id'].isin(common_ids)]

    # Criar diretório
    output_dir = "../data/processed/keywords"
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"keyword_{safe_keyword}.xlsx")

    # Garantir que abas não fiquem "invisíveis"
    def ensure_df(df, reference_df):
        if df.empty:
            return pd.DataFrame(columns=reference_df.columns)
        return df

    mises_only = ensure_df(mises_only, mises_citations_df)
    hayek_only = ensure_df(hayek_only, hayek_citations_df)

    # Para a aba conjunta, garantir colunas compatíveis
    if mises_and_hayek.empty:
        all_columns = list(set(mises_citations_df.columns).union(hayek_citations_df.columns))
        mises_and_hayek = pd.DataFrame(columns=all_columns)

    # Exportar
    with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
        mises_only.to_excel(writer, sheet_name='mises only', index=False)
        hayek_only.to_excel(writer, sheet_name='hayek only', index=False)
        mises_and_hayek.to_excel(writer, sheet_name='mises and hayek', index=False)

    print(f"Arquivo salvo em: {file_path}")

In [11]:
export_to_excel('abct', mises_citations_df, hayek_citations_df)
export_to_excel('praxeology', mises_citations_df, hayek_citations_df)
export_to_excel('money', mises_citations_df, hayek_citations_df)

Arquivo salvo em: ../data/processed/keywords\keyword_abct.xlsx
Arquivo salvo em: ../data/processed/keywords\keyword_praxeology.xlsx
Arquivo salvo em: ../data/processed/keywords\keyword_money.xlsx


In [12]:
import pandas as pd
import os
import ast

def export_by_part(mises_citations_df, hayek_citations_df):

    # ==================================================
    # 1️⃣ Normalizar + Explodir corretamente
    # ==================================================
    def normalize_and_explode(df):
        df = df.copy()

        if 'human_action_part_number' not in df.columns:
            raise ValueError("Coluna 'human_action_part_number' não encontrada")

        def to_list(x):
            # Caso já seja lista
            if isinstance(x, list):
                return x

            # Caso seja string tipo "['0','1']"
            if isinstance(x, str):
                try:
                    parsed = ast.literal_eval(x)
                    if isinstance(parsed, list):
                        return parsed
                    else:
                        return [parsed]
                except:
                    return [x]

            # Caso seja número ou outro tipo
            return [x]

        df['human_action_part_number'] = df['human_action_part_number'].apply(to_list)

        # 🔥 explode real
        df = df.explode('human_action_part_number')

        # limpar
        df['human_action_part_number'] = df['human_action_part_number'].astype(str).str.strip()

        df = df[
            (df['human_action_part_number'].notna()) &
            (df['human_action_part_number'] != 'nan') &
            (df['human_action_part_number'] != 'None') &
            (df['human_action_part_number'] != 'WHOLE')
        ]

        return df

    mises_df = normalize_and_explode(mises_citations_df)
    hayek_df = hayek_citations_df.copy()

    # ==================================================
    # 2️⃣ JOIN correto
    # ==================================================
    merged_df = pd.merge(
        mises_df,
        hayek_df,
        on='paragraph_id',
        how='outer',
        suffixes=('_mises', '_hayek'),
        indicator=True
    )

    # ==================================================
    # 3️⃣ PARTS corretos (agora escalares!)
    # ==================================================
    parts = sorted(mises_df['human_action_part_number'].unique())

    output_dir = "../data/processed/parts"
    os.makedirs(output_dir, exist_ok=True)

    for part in parts:

        safe_part = str(part)

        part_df = merged_df[
            merged_df['human_action_part_number'] == part
        ]

        mises_only = part_df[part_df['_merge'] == 'left_only']
        hayek_only = part_df[part_df['_merge'] == 'right_only']
        mises_and_hayek = part_df[part_df['_merge'] == 'both']

        # evitar abas vazias
        def ensure_df(df, reference_df):
            if df.empty:
                return pd.DataFrame(columns=reference_df.columns)
            return df

        mises_only = ensure_df(mises_only, mises_df)
        hayek_only = ensure_df(hayek_only, hayek_df)

        if mises_and_hayek.empty:
            all_columns = list(set(mises_df.columns).union(hayek_df.columns))
            mises_and_hayek = pd.DataFrame(columns=all_columns)

        file_path = os.path.join(output_dir, f"part_{safe_part}.xlsx")

        with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
            mises_only.to_excel(writer, sheet_name='mises only', index=False)
            hayek_only.to_excel(writer, sheet_name='hayek only', index=False)
            mises_and_hayek.to_excel(writer, sheet_name='mises and hayek', index=False)

        print(f"Arquivo salvo para part {part}: {file_path}")

In [ ]:
export_by_part(mises_citations_df, hayek_citations_df)


Arquivo salvo para part 0: ../data/processed/parts\part_0.xlsx
